In [54]:
import os
import json
import shutil
import yaml
from ultralytics import YOLO
import torch
import matplotlib.pyplot as plt
import random

In [55]:
# [2] Definizione della classe principale
class YOLOTrainer:
    def __init__(self, base_path):
        self.base_path = os.path.abspath(base_path)  # Usa percorso assoluto
        self.output_path = os.path.join(self.base_path, 'yolo_dataset')
        
        print(f"Base path: {self.base_path}")
        print(f"Output path: {self.output_path}")
        
        # Crea le cartelle necessarie
        for split in ['train', 'val']:
            for subdir in ['images', 'labels']:
                path = os.path.join(self.output_path, split, subdir)
                os.makedirs(path, exist_ok=True)
                print(f"Creata directory: {path}")

    def convert_coco_to_yolo(self, split='train', split_ratio=0.8):
        """
        Converte le annotazioni da COCO a YOLO
        """
        coco_path = os.path.join(self.base_path, '_annotations.coco.json')
        image_path = os.path.join(self.base_path)
        output_split_path = os.path.join(self.output_path, split)
        
        print(f"\nProcessing split: {split}")
        print(f"COCO annotations path: {coco_path}")
        print(f"Images path: {image_path}")
        print(f"Output path: {output_split_path}")
        
        # Verifica che il file delle annotazioni esista
        if not os.path.exists(coco_path):
            raise FileNotFoundError(f"File annotazioni non trovato: {coco_path}")
        
        # Carica le annotazioni COCO
        with open(coco_path, 'r') as f:
            coco_data = json.load(f)
        
        print(f"Caricate {len(coco_data['images'])} immagini e {len(coco_data['annotations'])} annotazioni")
        
        # Crea dizionario delle immagini e split
        images_dict = {img['id']: img for img in coco_data['images']}
        all_image_ids = list(images_dict.keys())
        
        # Imposta un seed per la riproducibilità
        random.seed(42)
        random.shuffle(all_image_ids)
        
        # Calcola l'indice di split
        split_idx = int(len(all_image_ids) * split_ratio)
        
        # Seleziona gli ID delle immagini per questo split
        if split == 'train':
            image_ids = all_image_ids[:split_idx]
        else:  # val
            image_ids = all_image_ids[split_idx:]
        
        print(f"Numero totale di immagini: {len(all_image_ids)}")
        print(f"Numero di immagini per {split}: {len(image_ids)}")
        
        # Raggruppa le annotazioni per immagine
        image_annotations = {}
        for ann in coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in image_annotations:
                image_annotations[img_id] = []
            image_annotations[img_id].append(ann)
        
        # Processa SOLO le immagini selezionate per questo split
        processed_images = 0
        for img_id in image_ids:  # Usa solo gli ID selezionati per questo split
            img_info = images_dict[img_id]
            src_img = os.path.join(image_path, img_info['file_name'])
            
            if not os.path.exists(src_img):
                print(f"ATTENZIONE: Immagine non trovata: {src_img}")
                continue
            
            # Copia l'immagine
            dst_img = os.path.join(output_split_path, 'images', img_info['file_name'])
            shutil.copy2(src_img, dst_img)
            
            # Crea il file di label
            if img_id in image_annotations:
                label_path = os.path.join(output_split_path, 'labels', 
                                        os.path.splitext(img_info['file_name'])[0] + '.txt')
                
                with open(label_path, 'w') as f:
                    for ann in image_annotations[img_id]:
                        bbox = ann['bbox']
                        x_center = (bbox[0] + bbox[2]/2) / img_info['width']
                        y_center = (bbox[1] + bbox[3]/2) / img_info['height']
                        width = bbox[2] / img_info['width']
                        height = bbox[3] / img_info['height']
                        f.write(f"0 {x_center} {y_center} {width} {height}\n")
            
            processed_images += 1
            if processed_images % (len(image_ids) // 10) == 0:
                print(f"Progresso: {processed_images}/{len(image_ids)} immagini processate ({processed_images/len(image_ids)*100:.1f}%)")
        
        # Verifica finale
        final_images = len(os.listdir(os.path.join(output_split_path, 'images')))
        final_labels = len(os.listdir(os.path.join(output_split_path, 'labels')))
        print(f"\nRiepilogo {split}:")
        print(f"Immagini copiate: {final_images}")
        print(f"Label create: {final_labels}")
        
        if final_images == 0:
            raise RuntimeError(f"Nessuna immagine copiata nel set {split}!")

    def create_yaml(self):
        yaml_content = {
            'path': self.output_path,
            'train': os.path.join(self.output_path, 'train/images'),
            'val': os.path.join(self.output_path, 'val/images'),
            'names': {0: 'gun'}
        }
        
        yaml_path = os.path.join(self.output_path, 'dataset.yaml')
        print(f"\nCreazione file YAML in: {yaml_path}")
        print("Contenuto YAML:")
        print(yaml_content)
        
        with open(yaml_path, 'w') as f:
            yaml.dump(yaml_content, f, default_flow_style=False)
        
        return yaml_path
    
    # [5] Metodo di training
    def train(self, epochs=100, batch_size=16, image_size=640):
        """
        Esegue il fine-tuning del modello
        """
        # Converti il dataset
        self.convert_coco_to_yolo('train')
        self.convert_coco_to_yolo('val')
        
        # Crea il file YAML
        yaml_path = self.create_yaml()
        
        # Inizializza il modello
        print("Inizializzazione modello YOLO...") 
        model = YOLO('yolov8n.pt')
        
        # Configura il training
        print("Avvio training...")
        results = model.train(
            data=yaml_path,
            epochs=epochs,
            imgsz=image_size,
            batch=batch_size,
            device='cuda' if torch.cuda.is_available() else 'cpu',
            patience=20,        # early stopping
            save=True,         # salva i migliori pesi
            verbose=True,
            augment=True,      # usa data augmentation
            degrees=10.0,      # rotazione massima
            translate=0.1,     # traslazione massima
            scale=0.5,         # scala massima
            fliplr=0.5,        # probabilità di flip orizzontale
            mosaic=1.0,        # probabilità di mosaic augmentation
            lr0=0.01,          # learning rate iniziale
            lrf=0.01,          # learning rate finale
            momentum=0.937,     # SGD momentum
            weight_decay=0.0005,# weight decay
            warmup_epochs=3,    # epoche di warmup
            warmup_momentum=0.8,
            warmup_bias_lr=0.1
        )
        
        return model, results

In [56]:
# [6] Funzione per visualizzare i risultati
def plot_results(results):
    """
    Visualizza i risultati del training
    """
    metrics = ['box_loss', 'cls_loss', 'dfl_loss']
    fig, axes = plt.subplots(1, len(metrics), figsize=(15, 5))
    
    for i, metric in enumerate(metrics):
        axes[i].plot(results.results_dict[metric])
        axes[i].set_title(metric)
        axes[i].grid(True)
    
    plt.tight_layout()
    plt.show()

In [57]:
# [7] Esecuzione del training
# Configura il path del dataset
base_path = 'Dataset/Images'

# Inizializza e esegui il training
trainer = YOLOTrainer(base_path)
model, results = trainer.train(epochs=100)

# Visualizza i risultati
plot_results(results)

Base path: /home/gianmarco/Scrivania/GunDetection/Dataset/Images
Output path: /home/gianmarco/Scrivania/GunDetection/Dataset/Images/yolo_dataset
Creata directory: /home/gianmarco/Scrivania/GunDetection/Dataset/Images/yolo_dataset/train/images
Creata directory: /home/gianmarco/Scrivania/GunDetection/Dataset/Images/yolo_dataset/train/labels
Creata directory: /home/gianmarco/Scrivania/GunDetection/Dataset/Images/yolo_dataset/val/images
Creata directory: /home/gianmarco/Scrivania/GunDetection/Dataset/Images/yolo_dataset/val/labels

Processing split: train
COCO annotations path: /home/gianmarco/Scrivania/GunDetection/Dataset/Images/annotations.coco.json
Images path: /home/gianmarco/Scrivania/GunDetection/Dataset/Images
Output path: /home/gianmarco/Scrivania/GunDetection/Dataset/Images/yolo_dataset/train


FileNotFoundError: File annotazioni non trovato: /home/gianmarco/Scrivania/GunDetection/Dataset/Images/annotations.coco.json